In [ ]:

import pandas as pd
import numpy as np

print("Entorno de ingeniería de datos listo")
print("Versión de Pandas:", pd.__version__)


In [ ]:


import pandas as pd
from pathlib import Path

# Ruta del archivo original
ruta = Path("netflix_titles.csv")

# Verificar que el archivo exista
if ruta.exists():
    df_raw = pd.read_csv(ruta, encoding="latin-1")

    print("Archivo cargado correctamente")
    print("Filas:", df_raw.shape[0])
    print("Columnas:", df_raw.shape[1])

    display(df_raw.head())
else:
    print("No se encontró el archivo CSV.")
    print("Carpeta actual:", Path.cwd())


In [ ]:

# Revisar los nombres de las columnas
print("Columnas del dataset:")
print(df_raw.columns.tolist())

# Identificar columnas completamente vacías
columnas_vacias = df_raw.columns[
    df_raw.isna().all()
].tolist()

print("\nColumnas completamente vacías:")
print(columnas_vacias)

print("\nCantidad de columnas vacías:", len(columnas_vacias))


In [ ]:

# Crear una copia para no modificar los datos originales
df = df_raw.copy()

# Eliminar columnas completamente vacías
df = df.dropna(axis=1, how="all")

# Verificar el resultado
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

display(df.head())


In [ ]:

# 1. Analizar valores nulos por columna
calidad = pd.DataFrame({
    "Valores nulos": df.isna().sum(),
    "Porcentaje nulos": (
        df.isna().mean() * 100
    ).round(2)
})

print("REPORTE DE VALORES NULOS")
display(calidad)

# 2. Revisar duplicados
print("REGISTROS DUPLICADOS")

print(
    "Filas completamente duplicadas:",
    df.duplicated().sum()
)

print(
    "Identificadores show_id duplicados:",
    df.duplicated(subset=["show_id"]).sum()
)

# 3. Revisar tipos de datos
print("TIPOS DE DATOS")
display(df.dtypes)


In [ ]:

# Revisar los tres registros documentados
ids_revisar = ["s5965", "s5966", "s5967"]

registros = df[
    df["show_id"].isin(ids_revisar)
]

display(registros[
    ["show_id", "type", "title", "release_year",
     "country", "listed_in"]
])

# Comprobar si existen en el CSV original
print("Registros encontrados:", len(registros))


In [ ]:

# Revisar los valores únicos de las columnas principales

print("TIPOS DE CONTENIDO")
print(df["type"].value_counts())

print("\nCLASIFICACIONES")
print(df["rating"].value_counts(dropna=False))

print("\nEJEMPLOS DE DURACIÓN")
print(df["duration"].dropna().unique()[:15])

print("\nRANGO DE AÑOS")
print("Mínimo:", df["release_year"].min())
print("Máximo:", df["release_year"].max())


In [ ]:

# Identificar registros con duraciones en la columna rating
ratings_sospechosos = ["74 min", "84 min", "66 min"]

registros_sospechosos = df[
    df["rating"].isin(ratings_sospechosos)
]

display(
    registros_sospechosos[
        ["show_id", "type", "title", "rating", "duration"]
    ]
)

print(
    "Registros sospechosos:",
    len(registros_sospechosos)
)


In [ ]:

# Crear una copia de trabajo
df_clean = df.copy()

# Identificar duraciones ubicadas en rating
mascara = (
    df_clean["rating"].astype("string")
    .str.fullmatch(r"\d+\s+min", na=False)
    & df_clean["duration"].isna()
)

print("Registros a corregir:", mascara.sum())

# Mover el valor a la columna correcta
df_clean.loc[mascara, "duration"] = (
    df_clean.loc[mascara, "rating"]
)

# Dejar rating como valor faltante
df_clean.loc[mascara, "rating"] = None

# Comprobar la corrección
display(
    df_clean.loc[
        mascara,
        ["show_id", "title", "rating", "duration"]
    ]
)


In [ ]:

# Separar duración en valor y unidad
duracion = df_clean["duration"].astype("string").str.extract(
    r"^\s*(\d+)\s+(.+?)\s*$"
)

# Convertir el valor a un tipo numérico
df_clean["duration_value"] = pd.to_numeric(
    duracion[0], errors="coerce"
).astype("Int64")

# Guardar la unidad de duración
df_clean["duration_unit"] = duracion[1].str.strip()

# Comprobar los resultados
display(
    df_clean[
        ["show_id", "type", "duration",
         "duration_value", "duration_unit"]
    ].head(10)
)

print(
    "Duraciones sin valor numérico:",
    df_clean["duration_value"].isna().sum()
)

print("\nUnidades encontradas:")
print(df_clean["duration_unit"].value_counts(dropna=False))


In [ ]:

# Estandarizar las unidades de duración
df_clean["duration_unit"] = (
    df_clean["duration_unit"]
    .replace({
        "Seasons": "Season"
    })
)

# Verificar el resultado
print("UNIDADES ESTANDARIZADAS")
print(
    df_clean["duration_unit"].value_counts(
        dropna=False
    )
)

# Validar que no haya valores sin duración
assert df_clean["duration_value"].notna().all(), \
    "Existen duraciones sin valor numérico"

print("\nValidación completada correctamente")


In [ ]:

# Seleccionar las columnas de la tabla principal
contenido = df_clean[
    [
        "show_id",
        "type",
        "title",
        "release_year",
        "rating",
        "duration_value",
        "duration_unit"
    ]
].copy()

# Validar la clave primaria
assert contenido["show_id"].notna().all()
assert contenido["show_id"].is_unique

# Verificar los resultados
print("TABLA CONTENIDO")
print("Registros:", len(contenido))
print("Columnas:", contenido.shape[1])

display(contenido.head())


In [ ]:

# Crear tabla de relación contenido-país
contenido_pais = df_clean[
    ["show_id", "country"]
].copy()

# Completar países faltantes
contenido_pais["country"] = (
    contenido_pais["country"]
    .fillna("Sin información")
)

# Separar países en filas independientes
contenido_pais["country"] = (
    contenido_pais["country"].str.split(",")
)

contenido_pais = contenido_pais.explode("country")

# Eliminar espacios sobrantes
contenido_pais["country"] = (
    contenido_pais["country"].str.strip()
)

# Eliminar relaciones duplicadas
contenido_pais = contenido_pais.drop_duplicates(
    subset=["show_id", "country"]
).reset_index(drop=True)

# Validaciones
assert contenido_pais["show_id"].isin(
    contenido["show_id"]
).all()

assert contenido_pais["country"].notna().all()

print("TABLA CONTENIDO_PAIS")
print("Registros:", len(contenido_pais))
print("Columnas:", contenido_pais.shape[1])

display(contenido_pais.head(10))


In [ ]:

# Crear tabla de relación contenido-género
contenido_genero = df_clean[
    ["show_id", "listed_in"]
].copy()

# Separar géneros en filas independientes
contenido_genero["listed_in"] = (
    contenido_genero["listed_in"].str.split(",")
)

contenido_genero = contenido_genero.explode("listed_in")

# Limpiar espacios y renombrar la columna
contenido_genero["listed_in"] = (
    contenido_genero["listed_in"].str.strip()
)

contenido_genero = contenido_genero.rename(
    columns={"listed_in": "genre"}
)

# Eliminar relaciones duplicadas
contenido_genero = contenido_genero.drop_duplicates(
    subset=["show_id", "genre"]
).reset_index(drop=True)

# Validar integridad referencial
assert contenido_genero["show_id"].isin(
    contenido["show_id"]
).all()

# Verificar resultados
print("TABLA CONTENIDO_GENERO")
print("Registros:", len(contenido_genero))
print("Columnas:", contenido_genero.shape[1])

display(contenido_genero.head(10))


In [ ]:

# Identificadores excluidos en la versión anterior
ids_revisar = ["s5965", "s5966", "s5967"]

# Revisar sus relaciones con países
paises_adicionales = contenido_pais[
    contenido_pais["show_id"].isin(ids_revisar)
]

# Revisar sus relaciones con géneros
generos_adicionales = contenido_genero[
    contenido_genero["show_id"].isin(ids_revisar)
]

print("RELACIONES DE PAÍSES")
display(paises_adicionales)
print("Cantidad:", len(paises_adicionales))

print("\nRELACIONES DE GÉNEROS")
display(generos_adicionales)
print("Cantidad:", len(generos_adicionales))


In [ ]:

# Identificadores ausentes en la versión anterior
ids_excluidos = ["s5965", "s5966", "s5967"]

# Relaciones de países sin esos tres contenidos
paises_comparables = contenido_pais[
    ~contenido_pais["show_id"].isin(ids_excluidos)
].copy()

print("RELACIONES DE PAÍSES COMPARABLES")
print("Pipeline actual:", len(paises_comparables))
print("MySQL anterior:", 10840)

print(
    "Diferencia:",
    len(paises_comparables) - 10840
)

# Comprobar duplicados
print(
    "Duplicados:",
    paises_comparables.duplicated(
        subset=["show_id", "country"]
    ).sum()
)


In [ ]:

# Leer la tabla de países del Excel anterior
archivo_excel = "netflix_dataset_final_normalizado.xlsx"

excel_paises = pd.read_excel(
    archivo_excel,
    sheet_name="contenido_pais"
)

# Estandarizar nombres y espacios
excel_paises.columns = excel_paises.columns.str.strip()

excel_paises["show_id"] = (
    excel_paises["show_id"].astype("string").str.strip()
)

excel_paises["country"] = (
    excel_paises["country"].astype("string").str.strip()
)

# Comparar las relaciones
comparacion = paises_comparables.merge(
    excel_paises[["show_id", "country"]].drop_duplicates(),
    on=["show_id", "country"],
    how="outer",
    indicator=True
)

print("RESULTADO DE LA COMPARACIÓN")
print(comparacion["_merge"].value_counts())

print("\nRELACIONES SOLO EN EL PIPELINE NUEVO")
display(
    comparacion.loc[
        comparacion["_merge"] == "left_only",
        ["show_id", "country"]
    ]
)

print("\nRELACIONES SOLO EN EL EXCEL ANTERIOR")
display(
    comparacion.loc[
        comparacion["_merge"] == "right_only",
        ["show_id", "country"]
    ]
)


In [ ]:

# Contar relaciones con países vacíos
vacios = (
    contenido_pais["country"]
    .astype("string")
    .str.strip()
    .eq("")
    .fillna(False)
)

print("Relaciones vacías encontradas:", vacios.sum())

# Eliminar únicamente relaciones vacías
contenido_pais = contenido_pais.loc[~vacios].copy()

# Restablecer índices
contenido_pais = contenido_pais.reset_index(drop=True)

# Validar que no queden países vacíos
assert (
    contenido_pais["country"]
    .astype("string")
    .str.strip()
    .ne("")
    .all()
)

print("Relaciones después de limpiar:", len(contenido_pais))
print("Validación completada")


In [ ]:

# Validar tabla contenido
assert contenido["show_id"].is_unique
assert contenido["show_id"].notna().all()

# Validar tabla contenido_pais
assert not contenido_pais.duplicated(
    subset=["show_id", "country"]
).any()

assert contenido_pais["show_id"].isin(
    contenido["show_id"]
).all()

assert contenido_pais["country"].notna().all()

# Validar tabla contenido_genero
assert not contenido_genero.duplicated(
    subset=["show_id", "genre"]
).any()

assert contenido_genero["show_id"].isin(
    contenido["show_id"]
).all()

assert contenido_genero["genre"].notna().all()

# Resumen final
print("VALIDACIÓN FINAL DEL PIPELINE")
print("----------------------------")
print("Contenido:", len(contenido))
print("Países:", len(contenido_pais))
print("Géneros:", len(contenido_genero))
print("\nTodas las validaciones fueron exitosas")


In [ ]:

%pip install sqlalchemy pymysql


In [ ]:

import sqlalchemy
import pymysql

print("SQLAlchemy:", sqlalchemy.__version__)
print("PyMySQL instalado correctamente")
print("Entorno listo para conectar con MySQL")


In [ ]:

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

# Solicitar contraseña del usuario de ETL
clave = getpass("Contraseña de netflix_etl: ")

# Configurar la conexión
url_etl = URL.create(
    drivername="mysql+pymysql",
    username="netflix_etl",
    password=clave,
    host="127.0.0.1",
    port=3306,
    database="netflix_etl_dev"
)

engine_etl = create_engine(
    url_etl,
    pool_pre_ping=True
)

# Verificar conexión
with engine_etl.connect() as conexion:
    resultado = conexion.execute(
        text("SELECT CURRENT_USER(), DATABASE()")
    ).fetchone()

    print("CONEXIÓN EXITOSA")
    print("Usuario:", resultado[0])
    print("Base de datos:", resultado[1])

del clave


In [ ]:

from sqlalchemy import text

# Crear tablas en la base de pruebas
sql_tablas = [
    """
    CREATE TABLE IF NOT EXISTS contenido (
        show_id VARCHAR(10) PRIMARY KEY,
        type VARCHAR(20) NOT NULL,
        title VARCHAR(500) NOT NULL,
        release_year SMALLINT,
        rating VARCHAR(30),
        duration_value SMALLINT,
        duration_unit VARCHAR(20)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS contenido_pais (
        show_id VARCHAR(10) NOT NULL,
        country VARCHAR(150) NOT NULL,
        PRIMARY KEY (show_id, country),
        FOREIGN KEY (show_id)
            REFERENCES contenido(show_id)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS contenido_genero (
        show_id VARCHAR(10) NOT NULL,
        genre VARCHAR(150) NOT NULL,
        PRIMARY KEY (show_id, genre),
        FOREIGN KEY (show_id)
            REFERENCES contenido(show_id)
    )
    """
]

with engine_etl.begin() as conexion:
    for consulta in sql_tablas:
        conexion.execute(text(consulta))

print("Las tres tablas fueron creadas correctamente")


In [ ]:

from sqlalchemy import text

with engine_etl.connect() as conexion:
    for tabla in [
        "contenido",
        "contenido_pais",
        "contenido_genero"
    ]:
        cantidad = conexion.execute(
            text(f"SELECT COUNT(*) FROM {tabla}")
        ).scalar()

        print(f"{tabla}: {cantidad} registros")


In [ ]:

from sqlalchemy import text

# Preparar los datos
datos_contenido = contenido[
    [
        "show_id", "type", "title",
        "release_year", "rating",
        "duration_value", "duration_unit"
    ]
].copy()

datos_pais = contenido_pais[
    ["show_id", "country"]
].copy()

datos_genero = contenido_genero[
    ["show_id", "genre"]
].copy()

# Comprobar que las tablas estén vacías
tablas = ["contenido", "contenido_pais", "contenido_genero"]

with engine_etl.connect() as conexion:
    cantidades = {
        tabla: conexion.execute(
            text(f"SELECT COUNT(*) FROM {tabla}")
        ).scalar()
        for tabla in tablas
    }

if any(cantidad != 0 for cantidad in cantidades.values()):
    print("Carga cancelada: alguna tabla ya contiene registros.")
else:
    # Cargar los datos en una transacción
    with engine_etl.begin() as conexion:
        datos_contenido.to_sql(
            "contenido",
            con=conexion,
            if_exists="append",
            index=False,
            chunksize=1000
        )

        datos_pais.to_sql(
            "contenido_pais",
            con=conexion,
            if_exists="append",
            index=False,
            chunksize=1000
        )

        datos_genero.to_sql(
            "contenido_genero",
            con=conexion,
            if_exists="append",
            index=False,
            chunksize=1000
        )

    print("CARGA ETL COMPLETADA CORRECTAMENTE")


In [ ]:

from sqlalchemy import text

print("VALIDACIÓN DE CARGA ETL")
print("-------------------------")

tablas = {
    "contenido": len(contenido),
    "contenido_pais": len(contenido_pais),
    "contenido_genero": len(contenido_genero)
}

with engine_etl.connect() as conexion:
    for tabla, esperado in tablas.items():
        obtenido = conexion.execute(
            text(f"SELECT COUNT(*) FROM {tabla}")
        ).scalar()

        estado = "OK" if obtenido == esperado else "ERROR"

        print(
            f"{tabla}: "
            f"Esperados={esperado} | "
            f"MySQL={obtenido} | "
            f"Estado={estado}"
        )


In [ ]:

from sqlalchemy import text
import pandas as pd

consultas = {
    "Paises sin contenido": """
        SELECT COUNT(*)
        FROM contenido_pais p
        LEFT JOIN contenido c
            ON p.show_id = c.show_id
        WHERE c.show_id IS NULL
    """,
    "Generos sin contenido": """
        SELECT COUNT(*)
        FROM contenido_genero g
        LEFT JOIN contenido c
            ON g.show_id = c.show_id
        WHERE c.show_id IS NULL
    """,
    "IDs duplicados en contenido": """
        SELECT COUNT(*) - COUNT(DISTINCT show_id)
        FROM contenido
    """,
    "Relaciones de pais duplicadas": """
        SELECT COUNT(*) - COUNT(DISTINCT show_id, country)
        FROM contenido_pais
    """,
    "Relaciones de genero duplicadas": """
        SELECT COUNT(*) - COUNT(DISTINCT show_id, genre)
        FROM contenido_genero
    """
}

resultados = []

with engine_etl.connect() as conexion:
    for validacion, consulta in consultas.items():
        errores = conexion.execute(
            text(consulta)
        ).scalar()

        resultados.append({
            "Validacion": validacion,
            "Errores": errores,
            "Estado": "OK" if errores == 0 else "REVISAR"
        })

reporte = pd.DataFrame(resultados)
display(reporte)


In [ ]:

import pandas as pd
from sqlalchemy import text

consulta = """
SELECT
    type AS tipo_contenido,
    COUNT(*) AS cantidad
FROM contenido
GROUP BY type
ORDER BY cantidad DESC
"""

with engine_etl.connect() as conexion:
    resumen = pd.read_sql(
        text(consulta),
        con=conexion
    )

display(resumen)


In [ ]:

# Verificar los componentes necesarios
componentes = {
    "Datos originales": "df_raw",
    "Tabla contenido": "contenido",
    "Tabla países": "contenido_pais",
    "Tabla géneros": "contenido_genero",
    "Conexión MySQL": "engine_etl"
}

for nombre, variable in componentes.items():
    estado = (
        "DISPONIBLE"
        if variable in globals()
        else "NO DISPONIBLE"
    )
    print(f"{nombre}: {estado}")


In [ ]:

def validar_datos(contenido, contenido_pais, contenido_genero):

    # Validar identificadores únicos
    assert contenido["show_id"].is_unique, (
        "Existen contenidos duplicados"
    )

    # Validar identificadores no nulos
    assert contenido["show_id"].notna().all(), (
        "Existen identificadores vacíos"
    )

    # Validar relaciones de países
    assert not contenido_pais.duplicated(
        subset=["show_id", "country"]
    ).any(), "Existen relaciones de países duplicadas"

    # Validar relaciones de géneros
    assert not contenido_genero.duplicated(
        subset=["show_id", "genre"]
    ).any(), "Existen relaciones de géneros duplicadas"

    # Validar integridad referencial
    assert contenido_pais["show_id"].isin(
        contenido["show_id"]
    ).all(), "Existen países sin contenido"

    assert contenido_genero["show_id"].isin(
        contenido["show_id"]
    ).all(), "Existen géneros sin contenido"

    # Validar valores obligatorios
    assert contenido_pais["country"].notna().all()
    assert contenido_genero["genre"].notna().all()

    assert contenido_pais["country"].astype(
        "string"
    ).str.strip().ne("").all()

    assert contenido_genero["genre"].astype(
        "string"
    ).str.strip().ne("").all()

    print("VALIDACIÓN AUTOMÁTICA EXITOSA")
    print("Contenidos:", len(contenido))
    print("Relaciones de países:", len(contenido_pais))
    print("Relaciones de géneros:", len(contenido_genero))


In [ ]:

validar_datos(
    contenido,
    contenido_pais,
    contenido_genero
)


In [ ]:

from pathlib import Path
import pandas as pd

def extraer_datos(ruta_archivo):
    """
    Lee el archivo CSV original de Netflix
    y devuelve un DataFrame sin modificarlo.
    """

    ruta = Path(ruta_archivo)

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}"
        )

    # Leer el archivo original
    datos = pd.read_csv(
        ruta,
        encoding="latin-1"
    )

    print("EXTRACCIÓN COMPLETADA")
    print("Filas:", datos.shape[0])
    print("Columnas:", datos.shape[1])

    return datos


In [ ]:

# Ejecutar extracción automática
datos_extraidos = extraer_datos(
    "netflix_titles.csv"
)

# Mostrar los primeros registros
display(datos_extraidos.head())


In [ ]:

def limpiar_datos_base(datos):
    """
    Realiza la limpieza inicial del dataset Netflix.
    """

    df = datos.copy()

    # Eliminar columnas sin información
    df = df.dropna(axis=1, how="all")

    # Detectar duraciones ubicadas incorrectamente
    patron = r"^\s*\d+\s*(?:min|Seasons?)\s*$"

    mascara = (
        df["duration"].isna()
        & df["rating"].astype("string").str.fullmatch(
            patron, case=False
        ).fillna(False)
    )

    # Corregir los registros identificados
    cantidad_corregida = int(mascara.sum())

    df.loc[mascara, "duration"] = df.loc[
        mascara, "rating"
    ]

    df.loc[mascara, "rating"] = None

    print("LIMPIEZA INICIAL COMPLETADA")
    print("Filas:", len(df))
    print("Columnas:", len(df.columns))
    print("Duraciones corregidas:", cantidad_corregida)

    return df


In [ ]:

datos_limpios = limpiar_datos_base(
    datos_extraidos
)


In [ ]:

def normalizar_duraciones(datos):
    """
    Separa la duración en valor numérico y unidad.
    """

    df = datos.copy()

    # Extraer la cantidad numérica
    df["duration_value"] = pd.to_numeric(
        df["duration"].astype("string").str.extract(
            r"^(\d+)", expand=False
        ),
        errors="coerce"
    )

    # Estandarizar la unidad de duración
    df["duration_unit"] = (
        df["duration"]
        .astype("string")
        .str.extract(
            r"^\d+\s*(min|Seasons?)$",
            expand=False
        )
        .replace({
            "Seasons": "Season"
        })
    )

    print("NORMALIZACIÓN DE DURACIONES")
    print(
        df.groupby(
            ["type", "duration_unit"]
        ).size()
    )

    print(
        "Duraciones sin valor:",
        df["duration_value"].isna().sum()
    )

    return df


In [ ]:

datos_normalizados = normalizar_duraciones(
    datos_limpios
)


In [ ]:

def crear_tablas_relacionales(datos):
    """
    Construye las tres tablas del modelo relacional.
    """

    # Tabla principal
    contenido_nuevo = datos[
        [
            "show_id",
            "type",
            "title",
            "release_year",
            "rating",
            "duration_value",
            "duration_unit"
        ]
    ].copy()

    # Tabla de países
    contenido_pais_nuevo = datos[
        ["show_id", "country"]
    ].copy()

    contenido_pais_nuevo["country"] = (
        contenido_pais_nuevo["country"]
        .fillna("Sin información")
        .str.split(",")
    )

    contenido_pais_nuevo = (
        contenido_pais_nuevo
        .explode("country")
    )

    contenido_pais_nuevo["country"] = (
        contenido_pais_nuevo["country"]
        .str.strip()
    )

    contenido_pais_nuevo = (
        contenido_pais_nuevo[
            contenido_pais_nuevo["country"].notna()
            & contenido_pais_nuevo["country"].ne("")
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Tabla de géneros
    contenido_genero_nuevo = datos[
        ["show_id", "listed_in"]
    ].copy()

    contenido_genero_nuevo["listed_in"] = (
        contenido_genero_nuevo["listed_in"]
        .str.split(",")
    )

    contenido_genero_nuevo = (
        contenido_genero_nuevo
        .explode("listed_in")
    )

    contenido_genero_nuevo = (
        contenido_genero_nuevo
        .rename(columns={"listed_in": "genre"})
    )

    contenido_genero_nuevo["genre"] = (
        contenido_genero_nuevo["genre"]
        .str.strip()
    )

    contenido_genero_nuevo = (
        contenido_genero_nuevo[
            contenido_genero_nuevo["genre"].notna()
            & contenido_genero_nuevo["genre"].ne("")
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print("TABLAS RELACIONALES CREADAS")
    print("Contenido:", len(contenido_nuevo))
    print("Países:", len(contenido_pais_nuevo))
    print("Géneros:", len(contenido_genero_nuevo))

    return (
        contenido_nuevo,
        contenido_pais_nuevo,
        contenido_genero_nuevo
    )


In [ ]:

(
    contenido_nuevo,
    contenido_pais_nuevo,
    contenido_genero_nuevo
) = crear_tablas_relacionales(
    datos_normalizados
)


In [ ]:

def ejecutar_pipeline_transformacion(ruta_archivo):
    """
    Ejecuta la extracción, limpieza,
    transformación y validación de Netflix.
    """

    print("INICIANDO PIPELINE ETL")
    print("----------------------")

    # 1. Extracción
    datos = extraer_datos(ruta_archivo)

    # 2. Limpieza inicial
    datos = limpiar_datos_base(datos)

    # 3. Normalización de duraciones
    datos = normalizar_duraciones(datos)

    # 4. Creación de tablas relacionales
    contenido_final, pais_final, genero_final = (
        crear_tablas_relacionales(datos)
    )

    # 5. Validación automática
    validar_datos(
        contenido_final,
        pais_final,
        genero_final
    )

    print("----------------------")
    print("PIPELINE DE TRANSFORMACIÓN COMPLETADO")

    return contenido_final, pais_final, genero_final


In [ ]:

(
    contenido_etl,
    pais_etl,
    genero_etl
) = ejecutar_pipeline_transformacion(
    "netflix_titles.csv"
)


In [ ]:

from sqlalchemy import text

def verificar_destino_carga(engine, contenido, pais, genero):
    """
    Comprueba la base de datos de destino
    y los registros existentes antes de cargar.
    """

    esperados = {
        "contenido": len(contenido),
        "contenido_pais": len(pais),
        "contenido_genero": len(genero)
    }

    with engine.connect() as conexion:

        base_actual = conexion.execute(
            text("SELECT DATABASE()")
        ).scalar()

        if base_actual != "netflix_etl_dev":
            raise ValueError(
                f"Base de datos no autorizada: {base_actual}"
            )

        print("BASE DE DATOS:", base_actual)
        print("----------------------------")

        for tabla, esperado in esperados.items():

            actual = conexion.execute(
                text(f"SELECT COUNT(*) FROM {tabla}")
            ).scalar()

            print(
                f"{tabla}: "
                f"MySQL={actual} | "
                f"Pipeline={esperado}"
            )

    print("----------------------------")
    print("VERIFICACIÓN DE DESTINO COMPLETADA")


In [ ]:

verificar_destino_carga(
    engine_etl,
    contenido_etl,
    pais_etl,
    genero_etl
)


In [ ]:

import pandas as pd
from sqlalchemy import text

def comparar_con_mysql(engine, tablas_pipeline):

    resultados = []

    with engine.connect() as conexion:

        base = conexion.execute(
            text("SELECT DATABASE()")
        ).scalar()

        if base != "netflix_etl_dev":
            raise ValueError(
                f"Base de datos no autorizada: {base}"
            )

        for nombre, datos_pipeline in tablas_pipeline.items():

            datos_mysql = pd.read_sql(
                text(f"SELECT * FROM {nombre}"),
                conexion
            )

            columnas = list(datos_pipeline.columns)

            # Comparar los datos sin depender del orden
            origen = datos_pipeline[columnas].copy()
            destino = datos_mysql[columnas].copy()

            for columna in columnas:
                if columna == "duration_value":
                    origen[columna] = pd.to_numeric(
                        origen[columna], errors="coerce"
                    )
                    destino[columna] = pd.to_numeric(
                        destino[columna], errors="coerce"
                    )
                else:
                    origen[columna] = origen[columna].astype("string")
                    destino[columna] = destino[columna].astype("string")

            claves = {
                "contenido": ["show_id"],
                "contenido_pais": ["show_id", "country"],
                "contenido_genero": ["show_id", "genre"]
            }[nombre]

            origen = origen.sort_values(
                claves
            ).reset_index(drop=True)

            destino = destino.sort_values(
                claves
            ).reset_index(drop=True)

            iguales = origen.equals(destino)

            resultados.append({
                "Tabla": nombre,
                "Registros": len(origen),
                "Coinciden": "SI" if iguales else "NO"
            })

    return pd.DataFrame(resultados)


In [ ]:

tablas_pipeline = {
    "contenido": contenido_etl,
    "contenido_pais": pais_etl,
    "contenido_genero": genero_etl
}

resultado_comparacion = comparar_con_mysql(
    engine_etl,
    tablas_pipeline
)

display(resultado_comparacion)


In [ ]:

# Consultar la tabla contenido desde MySQL
with engine_etl.connect() as conexion:
    contenido_mysql = pd.read_sql(
        text("SELECT * FROM contenido"),
        conexion
    )

# Preparar los dos DataFrames
origen = contenido_etl.copy()
destino = contenido_mysql.copy()

# Ordenar por identificador
origen = origen.sort_values(
    "show_id"
).reset_index(drop=True)

destino = destino.sort_values(
    "show_id"
).reset_index(drop=True)

# Comparar columna por columna
print("COMPARACIÓN POR COLUMNAS")
print("-----------------------")

for columna in origen.columns:

    valores_origen = origen[columna].astype("string")
    valores_destino = destino[columna].astype("string")

    diferencias = (
        valores_origen.fillna("<NULO>")
        != valores_destino.fillna("<NULO>")
    )

    print(
        f"{columna}: {diferencias.sum()} diferencias"
    )


In [ ]:

from pandas.testing import assert_frame_equal

# Preparar los datos
origen_final = origen.copy()
destino_final = destino.copy()

# Normalizar valores para la comparación
for columna in origen_final.columns:
    origen_final[columna] = (
        origen_final[columna]
        .astype("string")
        .fillna("<NULO>")
    )

    destino_final[columna] = (
        destino_final[columna]
        .astype("string")
        .fillna("<NULO>")
    )

# Verificar igualdad
try:
    assert_frame_equal(
        origen_final,
        destino_final,
        check_dtype=False
    )

    print("COMPARACIÓN EXITOSA")
    print("Los 8809 registros coinciden con MySQL.")

except AssertionError as error:
    print("SE DETECTARON DIFERENCIAS")
    print(error)


In [ ]:

from sqlalchemy import text

def cargar_datos(engine, contenido, pais, genero):
    """
    Carga los datos en MySQL únicamente
    cuando las tres tablas están vacías.
    """

    tablas = {
        "contenido": contenido,
        "contenido_pais": pais,
        "contenido_genero": genero
    }

    # Validar los datos antes de cargar
    validar_datos(contenido, pais, genero)

    with engine.begin() as conexion:

        # Verificar la base de datos
        base_actual = conexion.execute(
            text("SELECT DATABASE()")
        ).scalar()

        if base_actual != "netflix_etl_dev":
            raise ValueError(
                f"Base de datos no autorizada: {base_actual}"
            )

        # Comprobar si ya existen registros
        cantidades = {}

        for tabla in tablas:
            cantidades[tabla] = conexion.execute(
                text(f"SELECT COUNT(*) FROM {tabla}")
            ).scalar()

        if any(cantidad > 0 for cantidad in cantidades.values()):
            print("CARGA OMITIDA")
            print("Las tablas ya contienen registros.")
            print("No se modificó la base de datos.")
            return

        # Insertar respetando las relaciones
        for nombre, datos in tablas.items():
            datos.to_sql(
                name=nombre,
                con=conexion,
                if_exists="append",
                index=False,
                chunksize=1000
            )

        print("CARGA ETL COMPLETADA")
        print("Registros insertados:")
        for nombre, datos in tablas.items():
            print(f"{nombre}: {len(datos)}")


In [ ]:

cargar_datos(
    engine_etl,
    contenido_etl,
    pais_etl,
    genero_etl
)


In [ ]:

def ejecutar_pipeline_etl(ruta_archivo, engine):
    """
    Ejecuta el pipeline completo de Netflix:
    extracción, transformación, validación
    y carga a MySQL.
    """

    print("================================")
    print("INICIANDO PIPELINE ETL NETFLIX")
    print("================================")

    # 1. Extraer, transformar y validar
    contenido, pais, genero = (
        ejecutar_pipeline_transformacion(
            ruta_archivo
        )
    )

    # 2. Cargar en MySQL
    cargar_datos(
        engine,
        contenido,
        pais,
        genero
    )

    print("================================")
    print("EJECUCIÓN DEL PIPELINE FINALIZADA")
    print("================================")


In [ ]:

ejecutar_pipeline_etl(
    "netflix_titles.csv",
    engine_etl
)


In [ ]:

import sys
import pandas as pd
import sqlalchemy
import pymysql

print("TECNOLOGÍAS DEL PROYECTO")
print("------------------------")

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("PyMySQL:", pymysql.__version__)

with engine_etl.connect() as conexion:
    version_mysql = conexion.execute(
        text("SELECT VERSION()")
    ).scalar()

print("MySQL:", version_mysql)


In [ ]:

from pathlib import Path

dependencias = [
    f"pandas=={pd.__version__}",
    f"SQLAlchemy=={sqlalchemy.__version__}",
    f"PyMySQL=={pymysql.__version__}"
]

ruta_requirements = Path("requirements.txt")

ruta_requirements.write_text(
    "\n".join(dependencias) + "\n",
    encoding="utf-8"
)

print("ARCHIVO REQUIREMENTS CREADO")
print("---------------------------")
print(ruta_requirements.resolve())
print()
print(ruta_requirements.read_text(encoding="utf-8"))


In [ ]:

from pathlib import Path
import shutil

# Carpeta principal del proyecto
proyecto = Path.home() / "netflix-etl-pipeline"

# Crear las carpetas necesarias
carpetas = [
    "data",
    "notebooks",
    "src",
    "sql"
]

proyecto.mkdir(exist_ok=True)

for carpeta in carpetas:
    (proyecto / carpeta).mkdir(exist_ok=True)

# Copiar requirements.txt a la carpeta del proyecto
origen = Path("requirements.txt")
destino = proyecto / "requirements.txt"

shutil.copy2(origen, destino)

print("ESTRUCTURA DEL PROYECTO CREADA")
print("-----------------------------")
print("Ubicación:", proyecto)
print()

for elemento in sorted(proyecto.iterdir()):
    print(
        "[CARPETA]" if elemento.is_dir() else "[ARCHIVO]",
        elemento.name
    )


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

contenido_gitignore = """
# Credenciales y variables de entorno
.env
.env.*
!.env.example
*.pem
*.key

# Archivos temporales de Python
__pycache__/
*.pyc

# Entornos virtuales
.venv/
venv/

# Archivos temporales de Jupyter
.ipynb_checkpoints/

# Configuración del editor
.vscode/
.idea/

# Archivos de sistema
.DS_Store
Thumbs.db
"""

ruta_gitignore = proyecto / ".gitignore"

ruta_gitignore.write_text(
    contenido_gitignore.strip() + "\n",
    encoding="utf-8"
)

print("ARCHIVO .gitignore CREADO")
print("------------------------")
print("Ubicación:", ruta_gitignore)
print()
print(ruta_gitignore.read_text(encoding="utf-8"))


In [ ]:

from pathlib import Path
import shutil

# Ubicación del proyecto
proyecto = Path.home() / "netflix-etl-pipeline"

# Archivo original
origen_csv = Path("netflix_titles.csv")

# Destino dentro del proyecto
destino_csv = proyecto / "data" / "netflix_titles.csv"

# Comprobar que existe el archivo original
if not origen_csv.is_file():
    print("ERROR: No se encontró el archivo CSV original.")
else:
    # Copiar sin modificar el original
    shutil.copy2(origen_csv, destino_csv)

    print("ARCHIVO CSV COPIADO CORRECTAMENTE")
    print("--------------------------------")
    print("Ubicación:", destino_csv)
    print("Tamaño:", destino_csv.stat().st_size, "bytes")
